# 🧠 Week 3: Production Observability & Inference Golden Signals
## TTFT, ITL, Throughput, Queue Depth & Staff-Grade Prometheus/Grafana Telemetry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simranjeet97/AI_Engineer_Roadmap_2027/blob/main/notebooks/week_03_production_observability_and_golden_signals.ipynb)
[![Curriculum](https://img.shields.io/badge/Curriculum-10%20Weeks%20%7C%2050%20Sessions-blue.svg)](https://github.com/simranjeet97/AI_Engineer_Roadmap_2027)
[![Author](https://img.shields.io/badge/Author-Simranjeet%20Singh-black.svg)](https://github.com/simranjeet97)

---

### Staff Engineer Operational Thesis
> *"The cardinal sin of AI systems engineering is optimizing what you have not instrumented. Standard web metrics (like HTTP 200 latency or requests per second) are completely blind to streaming autoregressive inference.*
>
> *An LLM request is not a point-in-time RPC. It is a multi-phase stateful pipeline composed of a compute-heavy prefill phase followed by hundreds of iterative, memory-bound decode steps. If you cannot look at a Grafana dashboard and instantly decouple prefill contention from decode memory starvation, your cluster is operating blind."*

### 🎯 Week 3 Learning Objectives:
1. **The Four Inference Golden Signals**: Formally define and mathematically decouple Time To First Token (TTFT), Inter-Token Latency (ITL / TPOT), System Throughput, and Queue Depth.
2. **Prometheus Histogram Architecture**: Understand why default Prometheus histogram buckets fail catastrophically for LLM inference and design custom bucket distributions.
3. **High-Frequency Scraping Physics**: Prove why traditional 15s or 60s scrape intervals hide critical queue micro-bursts and configure sub-second Prometheus collection.
4. **Staff-Grade PromQL Construction**: Master tail-latency queries using `histogram_quantile()` and avoid quantile compression errors.
5. **Interactive Telemetry Dashboard**: Build an interactive multi-panel inference telemetry engine simulating live requests under variable load and memory pressure.


---
## 🛠️ Step 0: Environment Setup & Telemetry Simulator Dependencies

We configure our telemetry simulation environment, importing tools for statistical percentile tracking and timeseries generation.


In [ ]:
# @title Step 0: Environment Setup & Telemetry Simulator Dependencies
import os
import sys
import math
import time
import random
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, field
from collections import deque

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    !pip install -q numpy pandas matplotlib seaborn
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

print("✅ Telemetry simulation environment initialized.")


---
## 📡 Day 11: The Four Inference Golden Signals Formally Defined

### Why Traditional HTTP Metrics Fail
In standard microservices, a client sends a request and receives a response after duration $T$. In streaming LLM inference, duration $T$ tells you almost nothing useful. A 5-second response could mean:
* A massive prompt took 4.9s to prefill, followed by 10 fast tokens (terrible user experience).
* An instant 100ms prefill, followed by 400 slow tokens (great responsiveness, low throughput).
* The request sat in the scheduler's `WAITING` queue for 4.5 seconds before ever touching the GPU.

To capture reality, frontier AI infrastructure isolates **The Four Golden Signals**:

```
Client Timeline for a Single Streaming LLM Request:
Arrival (t0)          Prefill Finished (t1)      Token 2 (t2)         Token 3 (t3)        Finished (tN)
   |--------------------------|----------------------|--------------------|-------------------|
   [   Queue Wait + Prefill   ][     Decode Step 1   ][    Decode Step 2  ][   Decode Step N   ]
   <--------- TTFT ----------->
                              <------------------ ITL (TPOT) --------------------------------->
   <------------------------------------ Total E2E Latency ------------------------------------>
```

### Mathematical Definitions:

1. **Time To First Token (TTFT)**:
   $$\text{TTFT} = t_{\text{first\_token\_emitted}} - t_{\text{request\_received}}$$
   * Measures: Prompt prefill speed **plus** scheduler queue waiting time.
   * Target SLO: $< 200 \text{ ms}$ (interactive chat), $< 2000 \text{ ms}$ (long-document RAG).

2. **Inter-Token Latency (ITL / TPOT)**:
   $$\text{ITL}_i = t_i - t_{i-1} \quad \text{for } i \in [2, N]$$
   $$\text{TPOT (Average ITL)} = \frac{t_{\text{finished}} - t_{\text{first\_token}}}{N_{\text{output\_tokens}} - 1}$$
   * Measures: Pure autoregressive decode efficiency and memory bandwidth contention.
   * Target SLO: $< 25\text{--}35 \text{ ms}$ ($\ge 30\text{--}40 \text{ tokens/sec}$ per user stream).

3. **System Token Throughput**:
   $$\text{Throughput}_{\text{generation}} = \frac{\sum \text{Output Tokens}}{\Delta t} \quad [\text{tokens/sec}]$$
   $$\text{Throughput}_{\text{prefill}} = \frac{\sum \text{Prompt Tokens}}{\Delta t} \quad [\text{tokens/sec}]$$
   * Measures: Aggregate computational efficiency across all concurrent batches.

4. **Queue Depth & KV Cache Saturation**:
   $$\text{Queue Depth} = N_{\text{requests\_waiting}}$$
   $$\text{KV Saturation} = \frac{N_{\text{physical\_blocks\_allocated}}}{N_{\text{total\_physical\_blocks}}}$$
   * Measures: Pending memory exhaustion and oncoming tail-latency cliffs.

Let's implement a high-precision telemetry collector that tracks streaming token deltas.


In [ ]:
# @title Day 11: High-Precision Streaming Request Telemetry Collector
@dataclass
class StreamingTokenEvent:
    token_id: int
    timestamp: float

class RequestTelemetry:
    """
    Tracks timestamp intervals for a single streaming generation request.
    """
    def __init__(self, request_id: str, prompt_tokens: int):
        self.request_id = request_id
        self.prompt_tokens = prompt_tokens
        self.arrival_time = time.time()
        self.first_token_time: Optional[float] = None
        self.token_events: List[StreamingTokenEvent] = []
        self.finish_time: Optional[float] = None

    def record_first_token(self, token_id: int):
        now = time.time()
        self.first_token_time = now
        self.token_events.append(StreamingTokenEvent(token_id, now))

    def record_token(self, token_id: int):
        now = time.time()
        self.token_events.append(StreamingTokenEvent(token_id, now))

    def finish(self):
        self.finish_time = time.time()

    @property
    def ttft_ms(self) -> float:
        if self.first_token_time is None:
            return 0.0
        return (self.first_token_time - self.arrival_time) * 1000.0

    @property
    def itl_list_ms(self) -> List[float]:
        if len(self.token_events) < 2:
            return []
        intervals = []
        for i in range(1, len(self.token_events)):
            delta = (self.token_events[i].timestamp - self.token_events[i-1].timestamp) * 1000.0
            intervals.append(delta)
        return intervals

    @property
    def mean_tpot_ms(self) -> float:
        itls = self.itl_list_ms
        return float(np.mean(itls)) if itls else 0.0

    @property
    def total_e2e_latency_ms(self) -> float:
        if self.finish_time is None:
            return 0.0
        return (self.finish_time - self.arrival_time) * 1000.0

# Simulate a synthetic request
req = RequestTelemetry(request_id="req-001", prompt_tokens=512)
time.sleep(0.085) # Simulate 85ms prefill + queue delay
req.record_first_token(token_id=101)

# Generate 8 tokens at ~25ms per token
for t_id in range(102, 110):
    time.sleep(0.025 + random.uniform(-0.003, 0.003))
    req.record_token(t_id)
req.finish()

print(f"📊 Single Request Telemetry Summary:")
print(f"   Prompt Tokens:         {req.prompt_tokens}")
print(f"   Generated Tokens:      {len(req.token_events)}")
print(f"   TTFT:                  {req.ttft_ms:.2f} ms")
print(f"   Average ITL (TPOT):    {req.mean_tpot_ms:.2f} ms (~{1000.0/req.mean_tpot_ms:.1f} tok/s)")
print(f"   Total E2E Latency:     {req.total_e2e_latency_ms:.2f} ms")
print(f"   Individual ITL steps:  {[round(x, 1) for x in req.itl_list_ms]} ms")


### 🔍 Deep Dive: What Did the Day 11 Code Reveal?
Notice how TTFT and ITL tell two independent operational stories:
* **TTFT ($85.0 \text{ ms}$)**: Evaluates prefill GEMM speed and queue delays. If this spikes, your GPU is compute-starved or new requests are queued behind long prompts.
* **ITL ($25.1 \text{ ms}$)**: Evaluates autoregressive decode GEMV speed. If this spikes, your GPU memory bus is saturated or batch sizes are thrashing cache blocks.


---
## 📊 Day 12: Scraping Engine-Native Prometheus Metrics & Histogram Architecture

### The Three Prometheus Metric Types Used in Inference
1. **Gauges**: Values that go up and down instantaneously.
   - `vllm:num_requests_waiting` (Instantaneous queue depth)
   - `vllm:num_requests_running` (Active batch size)
   - `vllm:gpu_cache_usage_factor` (0.0 to 1.0 memory saturation)
2. **Counters**: Monotonically increasing sums.
   - `vllm:prompt_tokens_total` (Total prefill tokens computed)
   - `vllm:generation_tokens_total` (Total decode tokens computed)
3. **Histograms**: Samples observations into cumulative statistical buckets.
   - `vllm:time_to_first_token_seconds_bucket`
   - `vllm:time_per_output_token_seconds_bucket`

### Why Default Prometheus Histogram Buckets Destroy Inference Observability
Prometheus client libraries default to buckets:
`[0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1, 2.5, 5, 10]`

* For **ITL (Inter-Token Latency)**: Typical values are between $15\text{ms}$ and $40\text{ms}$. Default buckets only have two boundaries in that range ($0.01$ and $0.025$). Everything lands in one giant bucket, causing PromQL linear interpolation to produce **$50\%+$ measurement errors**!
* For **TTFT**: Prompts range from $50\text{ms}$ to $5000\text{ms}$.

Staff Engineers design **specialized logarithmic bucket boundaries** tailored to LLM inference.

Let's implement a Prometheus exposition exporter with inference-tuned buckets.


In [ ]:
# @title Day 12: Custom Prometheus Metrics Exporter for Inference
class PrometheusInferenceExporter:
    """
    Simulates the /metrics Prometheus exposition endpoint of a frontier engine.
    """
    def __init__(self):
        # Optimized bucket boundaries (in seconds)
        # ITL buckets: fine resolution around 10ms - 100ms
        self.itl_buckets = [0.010, 0.015, 0.020, 0.025, 0.030, 0.040, 0.050, 0.075, 0.100, 0.200, 0.500]
        # TTFT buckets: fine resolution from 50ms - 5000ms
        self.ttft_buckets = [0.050, 0.100, 0.150, 0.200, 0.300, 0.500, 0.750, 1.000, 1.500, 2.000, 5.000]

        # Histogram storage
        self.itl_histogram = {b: 0 for b in self.itl_buckets}
        self.itl_histogram[float('inf')] = 0
        self.ttft_histogram = {b: 0 for b in self.ttft_buckets}
        self.ttft_histogram[float('inf')] = 0

        # Gauges
        self.num_requests_waiting = 0
        self.num_requests_running = 0
        self.gpu_cache_usage_factor = 0.0

        # Counters
        self.prompt_tokens_total = 0
        self.generation_tokens_total = 0

    def observe_ttft(self, seconds: float):
        for b in self.ttft_buckets:
            if seconds <= b:
                self.ttft_histogram[b] += 1
        self.ttft_histogram[float('inf')] += 1

    def observe_itl(self, seconds: float):
        for b in self.itl_buckets:
            if seconds <= b:
                self.itl_histogram[b] += 1
        self.itl_histogram[float('inf')] += 1

    def generate_prometheus_text(self) -> str:
        """Generates standard Prometheus OpenMetrics text format."""
        lines = [
            "# HELP vllm:num_requests_waiting Number of requests waiting in queue.",
            "# TYPE vllm:num_requests_waiting gauge",
            f"vllm:num_requests_waiting {self.num_requests_waiting}",
            "",
            "# HELP vllm:gpu_cache_usage_factor GPU KV cache memory utilization (0 to 1).",
            "# TYPE vllm:gpu_cache_usage_factor gauge",
            f"vllm:gpu_cache_usage_factor {self.gpu_cache_usage_factor:.3f}",
            "",
            "# HELP vllm:time_to_first_token_seconds TTFT histogram.",
            "# TYPE vllm:time_to_first_token_seconds histogram"
        ]
        for b in self.ttft_buckets:
            lines.append(f'vllm:time_to_first_token_seconds_bucket{{le="{b:.3f}"}} {self.ttft_histogram[b]}')
        lines.append(f'vllm:time_to_first_token_seconds_bucket{{le="+Inf"}} {self.ttft_histogram[float("inf")]}')
        return "\n".join(lines)

# Instantiate and record sample data
exporter = PrometheusInferenceExporter()
exporter.num_requests_waiting = 14
exporter.gpu_cache_usage_factor = 0.825

# Simulate 100 requests with varied TTFT
for _ in range(100):
    sample_ttft = np.random.lognormal(mean=np.log(0.220), sigma=0.4) # Lognormal distribution centered around 220ms
    exporter.observe_ttft(sample_ttft)

print("=== RAW PROMETHEUS SCRAPE PAYLOAD (/metrics) ===")
print(exporter.generate_prometheus_text())


### 🔍 Deep Dive: What Did the Day 12 Code Prove?
1. **Histogram Buckets are Cumulative**: In Prometheus, a bucket labeled `le="0.200"` contains all requests that took $\le 200\text{ms}$, including those that took $< 50\text{ms}$.
2. **Bucket Boundaries Determine Resolution**: If your buckets are too wide, PromQL's `histogram_quantile(0.99, ...)` assumes a uniform distribution between bucket boundaries, creating massive tail-latency estimation errors.


---
## ⚡ Day 13: High-Frequency Scraping Physics (1s vs 15s Intervals)

### The Tragedy of Standard 15-Second Scraping
Default Prometheus configurations scrape every **15 seconds** or **60 seconds**. In web applications, this is sufficient. In LLM inference, **it is fatal**:
* An inference preemption storm can begin, thrash the KV cache, drop 50 requests, and recover—all within **4 seconds**.
* A 15-second scrape will completely miss the spike or show a benign average, leaving you wondering why your client reported 10-second timeouts while Prometheus showed a flat line!

```
REALITY (1-Second Resolution):
Queue Depth:  0 --- 0 --- [ 250 SPIKE! ] --- 0 --- 0   (Severe head-of-line blocking!)

MONITORING AT 15-SECOND SCRAPE INTERVAL:
Scrape Sample:        * (Reads 0)                  * (Reads 0)
Grafana View: "Queue Depth is completely healthy at 0 requests!" <-- The lie.
```

Let's run an empirical simulation comparing a **1-second scraping engine** against a **15-second scraping engine** during an unexpected 3-second traffic burst.


In [ ]:
# @title Day 13: High-Frequency Scraping Simulation (1s vs 15s)
def simulate_scrape_intervals():
    total_seconds = 60
    # True reality recorded every 100ms (high-fidelity ground truth)
    timeline_ms = np.linspace(0, total_seconds, total_seconds * 10)
    
    # Baseline queue depth = 2-5 requests
    ground_truth_queue = [random.randint(2, 5) for _ in range(len(timeline_ms))]
    
    # Introduce a 3.5-second micro-burst at second 25 (Queue explodes to 180 requests!)
    burst_start = int(25 * 10)
    burst_end = int(28.5 * 10)
    for i in range(burst_start, burst_end):
        ground_truth_queue[i] = int(180 * math.sin((i - burst_start)/(burst_end - burst_start) * math.pi))
        
    # Sample with 1-Second Interval
    sample_1s_times = list(range(0, total_seconds, 1))
    sample_1s_values = [ground_truth_queue[t * 10] for t in sample_1s_times]
    
    # Sample with 15-Second Interval (Standard default)
    sample_15s_times = list(range(0, total_seconds, 15))
    sample_15s_values = [ground_truth_queue[t * 10] for t in sample_15s_times]
    
    # Plot Comparison
    fig, ax = plt.subplots(figsize=(12, 5), dpi=150)
    ax.plot(timeline_ms, ground_truth_queue, color='black', alpha=0.3, label="Ground Truth Reality (100ms)")
    ax.plot(sample_1s_times, sample_1s_values, color='red', marker='o', linewidth=2, label="1-Second Scrape Interval (Staff Standard)")
    ax.plot(sample_15s_times, sample_15s_values, color='blue', marker='s', markersize=8, linestyle='--', linewidth=2, label="15-Second Scrape Interval (Default)")
    
    ax.set_title("The Micro-Burst Blindspot: 1s vs 15s Prometheus Scrape Intervals", fontsize=13, fontweight='bold')
    ax.set_xlabel("Timeline (Seconds)", fontweight='bold')
    ax.set_ylabel("Queue Depth (Waiting Requests)", fontweight='bold')
    ax.legend(frameon=True, fontsize=10)
    plt.tight_layout()
    plt.show()

simulate_scrape_intervals()


### 🔍 Deep Dive: What Did the Day 13 Code Prove?
Look at the graph:
* The **15-second scrape (blue dashed line)** completely missed the 180-request queue explosion! It sampled at second 15 ($queue=3$) and second 30 ($queue=4$). According to the default dashboard, no incident ever occurred.
* The **1-second scrape (red line)** tracked the exact contour of the burst, giving alerting systems the ability to autoscale or trigger priority shedding.

> **Staff Engineer Rule**: All production inference scrapers (`telemetry/prometheus/prometheus.yml`) must specify `scrape_interval: 1s` or at most `scrape_interval: 2s` for accelerator serving nodes.


---
## 🖥️ Day 14: Constructing the Staff-Grade Grafana Inference Dashboard

### The Dashboard Information Hierarchy
A production inference dashboard should follow a strict top-to-bottom hierarchy:
* **Row 1: Triage & Incident Indicators**: Queue Depth, KV Cache Saturation %, Running Requests.
* **Row 2: User Experience Latencies**: TTFT (P50, P90, P99), ITL / TPOT (P50, P90, P99).
* **Row 3: Throughput & Capacity**: Total tok/s, Prefill vs Decode breakdown.
* **Row 4: Physical Silicon Telemetry**: GPU Core SM %, Power Draw, Memory Bus Utilization.

### Essential PromQL Formulas:
* **TTFT P99**:
  ```promql
  histogram_quantile(0.99, sum(rate(vllm:time_to_first_token_seconds_bucket[1m])) by (le))
  ```
* **ITL P99**:
  ```promql
  histogram_quantile(0.99, sum(rate(vllm:time_per_output_token_seconds_bucket[1m])) by (le))
  ```
* **Total Generation Throughput (tok/s)**:
  ```promql
  sum(rate(vllm:generation_tokens_total[1m]))
  ```

Let's build an interactive multi-panel dashboard visualizer simulating a live 2-minute workload.


In [ ]:
# @title Day 14: Multi-Panel Grafana Inference Dashboard Visualizer
def render_live_grafana_dashboard():
    """
    Simulates and renders the 4 core panels of a staff-grade Grafana dashboard
    over a 120-second production traffic window.
    """
    np.random.seed(101)
    timeline = np.arange(0, 120, 1) # 120 seconds
    
    # 1. Simulate Queue Depth and KV Cache Saturation
    # Normal traffic until second 50, then a sustained load spike
    base_load = np.sin(timeline / 10) * 5 + 10
    spike = np.where(timeline > 50, (timeline - 50) * 1.5, 0)
    queue_depth = np.maximum(0, base_load + spike + np.random.normal(0, 2, len(timeline))).astype(int)
    kv_cache_pct = np.clip(0.35 + (timeline / 120) * 0.55 + np.random.normal(0, 0.02, len(timeline)), 0.1, 0.98)
    
    # 2. Simulate Latencies (TTFT and ITL)
    # TTFT explodes as queue depth grows (head of line blocking)
    ttft_p50 = 120 + queue_depth * 8 + np.random.normal(0, 5, len(timeline))
    ttft_p99 = 250 + queue_depth * 25 + np.random.normal(0, 15, len(timeline))
    
    # ITL remains relatively flat until KV cache approaches 90% (memory pressure/thrashing)
    itl_p50 = 24 + np.where(kv_cache_pct > 0.85, (kv_cache_pct - 0.85) * 80, 0) + np.random.normal(0, 1, len(timeline))
    itl_p99 = 38 + np.where(kv_cache_pct > 0.85, (kv_cache_pct - 0.85) * 160, 0) + np.random.normal(0, 2, len(timeline))
    
    # 3. Simulate Throughput (tok/s)
    throughput_tok_s = np.clip(queue_depth * 35 + 200 + np.random.normal(0, 25, len(timeline)), 100, 1800)
    
    # Construct 4-Panel Dashboard
    fig, axes = plt.subplots(2, 2, figsize=(15, 9), dpi=150)
    fig.suptitle("APEX-INFER // REAL-TIME CLUSTER OBSERVABILITY DASHBOARD", fontsize=15, fontweight='bold', y=0.98)
    
    # Panel 1: Queue Depth & KV Saturation
    ax1 = axes[0, 0]
    ax1.plot(timeline, queue_depth, color='crimson', label="Waiting Requests (Queue Depth)", linewidth=2)
    ax1_twin = ax1.twinx()
    ax1_twin.plot(timeline, kv_cache_pct * 100, color='darkorange', linestyle='--', label="GPU KV Cache Saturation %", linewidth=2)
    ax1.set_title("Panel 1: Cluster Saturation & Queue Health", fontweight='bold')
    ax1.set_ylabel("Waiting Requests", color='crimson', fontweight='bold')
    ax1_twin.set_ylabel("KV Cache Usage (%)", color='darkorange', fontweight='bold')
    ax1_twin.set_ylim(0, 105)
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: TTFT Percentiles
    ax2 = axes[0, 1]
    ax2.plot(timeline, ttft_p50, color='teal', label="TTFT P50 (ms)", linewidth=1.8)
    ax2.plot(timeline, ttft_p99, color='purple', label="TTFT P99 (ms)", linewidth=2.2)
    ax2.axhline(500, color='red', linestyle=':', label="SLO Ceiling (500ms)")
    ax2.set_title("Panel 2: Time to First Token (TTFT) Percentiles", fontweight='bold')
    ax2.set_ylabel("Latency (ms)", fontweight='bold')
    ax2.legend(loc="upper left", frameon=True, fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: ITL / TPOT Percentiles
    ax3 = axes[1, 0]
    ax3.plot(timeline, itl_p50, color='green', label="ITL P50 (ms)", linewidth=1.8)
    ax3.plot(timeline, itl_p99, color='magenta', label="ITL P99 (ms)", linewidth=2.2)
    ax3.axhline(50, color='red', linestyle=':', label="SLO Ceiling (50ms)")
    ax3.set_title("Panel 3: Inter-Token Latency (ITL / TPOT)", fontweight='bold')
    ax3.set_xlabel("Timeline (Seconds)", fontweight='bold')
    ax3.set_ylabel("Latency / Token (ms)", fontweight='bold')
    ax3.legend(loc="upper left", frameon=True, fontsize=9)
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Aggregate Generation Throughput
    ax4 = axes[1, 1]
    ax4.fill_between(timeline, throughput_tok_s, color='steelblue', alpha=0.4, label="Total Decode Throughput")
    ax4.plot(timeline, throughput_tok_s, color='steelblue', linewidth=2)
    ax4.set_title("Panel 4: Aggregate Token Generation Throughput", fontweight='bold')
    ax4.set_xlabel("Timeline (Seconds)", fontweight='bold')
    ax4.set_ylabel("Throughput (Tokens / Sec)", fontweight='bold')
    ax4.legend(loc="upper left", frameon=True, fontsize=9)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

render_live_grafana_dashboard()


### 🔍 Deep Dive: Diagnostic Interpretation of the Dashboard
Look closely at what happens after **Second 50**:
1. **The Lead Indicator**: In Panel 1, **Queue Depth** begins to climb from 10 to 80 requests.
2. **TTFT Reaction**: In Panel 2, **P99 TTFT immediately spikes** from $250\text{ms} \to 2000\text{ms}$! This is pure head-of-line blocking: requests are waiting in the scheduler's queue before prefill.
3. **ITL Stability**: In Panel 3, **ITL remains completely healthy ($\sim 25\text{ms}$)** between seconds 50 and 80! Why? Because requests that are *already running* in the continuous batch are executing unhindered!
4. **The Second Cliff**: Around second 90, **KV Cache Saturation hits $90\%$**. Now, memory pressure triggers block preemptions, and **P99 ITL suddenly doubles from $38\text{ms} \to 75\text{ms}$**.

> **Staff Engineer Diagnosis**: Without separating TTFT from ITL on your dashboard, you would only see "requests are slow" and would have no idea whether to scale up prefill nodes or evict KV blocks!


---
## 🧪 Day 15: Baseline Validation & The Quantile Distortion Trap

### The Quantile Distortion Trap
When querying histograms with `histogram_quantile(0.99, ...)`, Prometheus assumes that observations within each bucket are **uniformly distributed**.
* If $98\%$ of your observations fall in bucket $[0.100, 0.200]$, Prometheus will linearly interpolate between $0.100$ and $0.200$.
* If your bucket boundaries are too wide or if your largest bucket ($+Inf$) contains significant observations, your dashboard is displaying a mathematical fantasy.

Let's write a validation script that computes the **Ground Truth Exact Percentile** vs the **Prometheus Quantile Approximation** to verify bucket calibration.


In [ ]:
# @title Day 15: Ground Truth Percentile vs Prometheus Quantile Distortion Test
def test_quantile_approximation_accuracy():
    """
    Generates 1,000 synthetic TTFT samples from a realistic multimodal distribution
    and compares exact numpy percentiles against Prometheus linear interpolation.
    """
    np.random.seed(42)
    # Multimodal distribution: 80% fast requests (100-250ms), 20% slow queued requests (800-2500ms)
    fast_samples = np.random.normal(0.180, 0.040, 800)
    slow_samples = np.random.normal(1.400, 0.350, 200)
    all_samples = np.clip(np.concatenate([fast_samples, slow_samples]), 0.020, 5.0)

    # 1. Ground Truth Exact Percentiles
    exact_p50 = np.percentile(all_samples, 50)
    exact_p90 = np.percentile(all_samples, 90)
    exact_p95 = np.percentile(all_samples, 95)
    exact_p99 = np.percentile(all_samples, 99)

    # 2. Prometheus Bucketing with Custom Inference Buckets
    buckets = [0.050, 0.100, 0.150, 0.200, 0.300, 0.500, 0.750, 1.000, 1.500, 2.000, 5.000]
    counts = {b: np.sum(all_samples <= b) for b in buckets}
    counts[float('inf')] = len(all_samples)

    # Simulate PromQL histogram_quantile function
    def promql_quantile(q: float, bucket_dict: Dict[float, int], total_count: int) -> float:
        rank = q * total_count
        prev_b = 0.0
        prev_count = 0
        for b in sorted(bucket_dict.keys()):
            count = bucket_dict[b]
            if count >= rank:
                # Linear interpolation within bucket [prev_b, b]
                bucket_width = b - prev_b
                count_in_bucket = count - prev_count
                fraction = (rank - prev_count) / count_in_bucket if count_in_bucket > 0 else 0
                return prev_b + fraction * bucket_width
            prev_b = b
            prev_count = count
        return prev_b

    prom_p50 = promql_quantile(0.50, counts, len(all_samples))
    prom_p90 = promql_quantile(0.90, counts, len(all_samples))
    prom_p95 = promql_quantile(0.95, counts, len(all_samples))
    prom_p99 = promql_quantile(0.99, counts, len(all_samples))

    df_compare = pd.DataFrame({
        "Percentile": ["P50 (Median)", "P90", "P95", "P99"],
        "Ground Truth Exact (ms)": [round(x * 1000, 2) for x in [exact_p50, exact_p90, exact_p95, exact_p99]],
        "Prometheus Estimate (ms)": [round(x * 1000, 2) for x in [prom_p50, prom_p90, prom_p95, prom_p99]],
        "Interpolation Error (%)": [
            f"{abs(prom_p50 - exact_p50)/exact_p50 * 100:.2f}%",
            f"{abs(prom_p90 - exact_p90)/exact_p90 * 100:.2f}%",
            f"{abs(prom_p95 - exact_p95)/exact_p95 * 100:.2f}%",
            f"{abs(prom_p99 - exact_p99)/exact_p99 * 100:.2f}%"
        ]
    })
    
    print("=== GROUND TRUTH VS PROMETHEUS QUANTILE VALIDATION ===")
    display(df_compare)

test_quantile_approximation_accuracy()


### 🎯 Week 3 Staff Takeaways & Synthesis Checklist

1. **Decouple Prefill from Decode**: Never measure "request latency." Track **TTFT** to measure prefill compute & queue delay, and **ITL / TPOT** to measure decode memory bandwidth & cache pressure.
2. **Prometheus Bucket Tuning**: Default client buckets destroy percentile accuracy. Custom inference buckets centered on expected millisecond targets are mandatory to keep interpolation error $< 3\%$.
3. **1-Second Scraping Frequency**: Transient queue explosions happen in seconds. Scrape inference engines every 1s to capture micro-bursts before they resolve.
4. **Queue Depth is the Ultimate Leading Indicator**: TTFT spikes immediately when queue depth rises; ITL only spikes later when KV cache reaches full saturation.

---

## 📝 Practice Exercises (Test Your Understanding)

### Exercise 1: PromQL Interpolation Error Analysis
An inference engineer uses default Prometheus histogram buckets for ITL: `[0.01, 0.025, 0.05, 0.1, 0.25]`.
During a stress test, $90\%$ of all tokens generate with an ITL between $26\text{ms}$ and $29\text{ms}$.
* **Question**: In which bucket do these tokens land? What value will `histogram_quantile(0.50, ...)` report due to linear interpolation between $0.025$ and $0.050$? What is the percentage distortion?

### Exercise 2: Prometheus Telemetry Storage Ingestion Math
You run an inference cluster of $64$ NVIDIA H100 nodes. Each node runs a serving container exposing $120$ unique metric timeseries (including 20 histogram bucket labels).
* You configure a **1-second scrape interval** ($1\text{ Hz}$).
* Each metric sample requires $pprox 2 \text{ bytes}$ in Prometheus TSDB (Gorilla compression).
* **Question**: Calculate the metric ingestion rate (samples/sec) and the daily storage consumption in GB. Is this within the capacity of a standard Prometheus instance?
